In [ ]:
#%pip install pandas pyyaml roboflow

  Using cached pandas-3.1.0-cp313-cp313-win_amd64.whl.metadata (18 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.5.1-cp313-cp313-win_amd64.whl.metadata (5.2 kB)
  Using cached matplotlib-3.11.2-cp313-cp313-win_amd64.whl.metadata (80 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached urllib3-2.8.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached annotated_doc-0.0.5-py3-none-any.w

In [5]:
from pathlib import Path

import pandas as pd
import yaml
from roboflow import Roboflow

from api_key import load_api_key

In [6]:
WORKSPACE = "dataset-w8sqm"
PROJECT = "tomato-ripeness-jionb"
VERSION = 14
DATASET_DIR = Path("datasets") / f"tomato-ripeness-v{VERSION}"
SPLITS = ["train", "valid", "test"]

In [7]:
rf = Roboflow(api_key=load_api_key())
rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov11", location=str(DATASET_DIR))
names = yaml.safe_load((DATASET_DIR / "data.yaml").read_text())["names"]
names

loading Roboflow workspace...
loading Roboflow project...


['half-ripe', 'not_ripe', 'ripe']

In [8]:
images = pd.DataFrame(
    {"split": split, "image": path.stem}
    for split in SPLITS
    for path in (DATASET_DIR / split / "images").iterdir()
)
images["photo"] = images["image"].str.replace(r"\.rf\.[0-9a-f]+$", "", regex=True)
len(images)

1012

In [9]:
splits = images.groupby("split").agg(images=("image", "size"), photos=("photo", "nunique")).reindex(SPLITS)
splits["share"] = (splits["images"] / splits["images"].sum()).round(3)
splits.loc["total"] = [splits["images"].sum(), images["photo"].nunique(), 1.0]
splits.astype({"images": int, "photos": int})

,images,photos,share
split,,,
train,705,235,0.697
valid,201,200,0.199
test,106,106,0.105
total,1012,540,1.000


In [10]:
labels = pd.DataFrame(
    {"split": split, "image": path.stem, "class": names[int(line.split()[0])]}
    for split in SPLITS
    for path in (DATASET_DIR / split / "labels").glob("*.txt")
    for line in path.read_text().splitlines()
    if line.strip()
)
pd.crosstab(labels["class"], labels["split"], margins=True, margins_name="total")[SPLITS + ["total"]]

split,train,valid,test,total
class,,,,
half-ripe,990,676,309,1975
not_ripe,1416,577,450,2443
ripe,2313,949,412,3674
total,4719,2202,1171,8092
